# Introduction to LSTM with 3D Input Tensors: Synthetic DNA Sequence Classification

Welcome to this hands-on Google Colab practical! In this notebook, we will build a small sequence-classification model using a **Long Short-Term Memory (LSTM)** network and a **3D input tensor**.

We will use **synthetic biological sequence data** so you can see how an LSTM learns sequential patterns without needing a massive genomics dataset or a GPU.

### Practical Overview & Suggested Classroom Timing (30 Minutes)
1. **Understand 3D Input Tensors** *(5 min)*
2. **Understand What an LSTM Does & Its Three Gates** *(5 min)*
3. **Generate & One-Hot Encode DNA Sequences** *(5 min)*
4. **Build the LSTM Model Architecture** *(5 min)*
5. **Train the Model** *(5 min)*
6. **Evaluate Predictions, Visualize Results & Classroom Review** *(5 min)*

---
## 1. Understanding the 3D Input Tensor

An LSTM layer expects input data in a 3-dimensional format:

$$\boxed{\text{Samples} \times \text{Timesteps} \times \text{Features}}$$

For example, suppose an input shape is `(200, 20, 4)`:

| Dimension | Value | Meaning |
| :--- | :--- | :--- |
| **Samples** | `200` | 200 individual biological sequences |
| **Timesteps** | `20` | Each sequence has 20 positions (nucleotides) |
| **Features** | `4` | Four numerical features at each position |

Think of it this way:
- **Sample** = one biological sequence.
- **Timestep** = one position in that sequence.
- **Features** = numerical information associated with that position.

### Representing DNA with One-Hot Encoding
For a DNA sequence, we can represent each nucleotide (`A`, `T`, `G`, `C`) using **one-hot encoding**:

| Nucleotide | A | T | G | C |
| :--- | :---: | :---: | :---: | :---: |
| **A** | `1` | `0` | `0` | `0` |
| **T** | `0` | `1` | `0` | `0` |
| **G** | `0` | `0` | `1` | `0` |
| **C** | `0` | `0` | `0` | `1` |

Therefore:
- A single DNA sequence containing 20 nucleotides becomes a 2D matrix of shape `(20, 4)`.
- A batch of 200 such sequences becomes a **3D input tensor** of shape `(200, 20, 4)`.

---
## 2. What Does an LSTM Actually Do?

An **LSTM (Long Short-Term Memory)** is a type of recurrent neural network (RNN) designed to learn patterns in sequential data. Unlike a standard dense neural network that treats features as independent inputs, an LSTM processes a sequence **step by step**:

$$\text{A} \rightarrow \text{T} \rightarrow \text{G} \rightarrow \text{C} \rightarrow \text{G} \rightarrow \text{T} \rightarrow \text{A} \rightarrow \text{C}$$

At each position (timestep), the LSTM:
1. Receives the current nucleotide.
2. Considers information from previous positions.
3. Updates its internal cell memory.
4. Passes information forward to the next position.

### The Three Gates of an LSTM
| Gate | Function | Simple Explanation |
| :--- | :--- | :--- |
| **Forget Gate** | Controls what information to discard from cell memory | *"What should I forget?"* |
| **Input Gate** | Controls what new information to store in memory | *"What should I remember?"* |
| **Output Gate** | Controls what information to pass forward as output | *"What should I communicate?"* |

**Key Concept:** An LSTM can retain relevant information from earlier positions while processing later positions (for instance, remembering a nucleotide pattern at position 5 when interpreting position 20).

---
## 3. Hands-On Practical: Train an LSTM on Synthetic DNA Data

### Objective
Train an LSTM to distinguish DNA sequences containing a specific nucleotide motif (`ATGCGT`) from sequences that do not contain that motif—**regardless of where the motif appears** in the sequence.

> *Note: This is a teaching demonstration, not a real biological prediction task.*

### Cell 1: Import Libraries

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

### Cell 2: Generate Synthetic DNA Sequences

We will generate `2,000` random DNA sequences, each of length `30` nucleotides:
- **Label `1` (Positive sample):** The motif `ATGCGT` is inserted at a random position in the sequence.
- **Label `0` (Negative sample):** Random sequence guaranteed **not** to contain `ATGCGT`.

In [ ]:
DNA_BASES = ["A", "T", "G", "C"]

SEQUENCE_LENGTH = 30
NUMBER_OF_SAMPLES = 2000
MOTIF = "ATGCGT"


def generate_random_sequence(length):
    return "".join(np.random.choice(DNA_BASES, size=length))


sequences = []
labels = []

for i in range(NUMBER_OF_SAMPLES):
    sequence = generate_random_sequence(SEQUENCE_LENGTH)

    if i % 2 == 0:
        # Positive sample: insert the motif at a random valid start index
        position = np.random.randint(0, SEQUENCE_LENGTH - len(MOTIF) + 1)
        sequence = (
            sequence[:position]
            + MOTIF
            + sequence[position + len(MOTIF):]
        )
        label = 1
    else:
        # Negative sample: reject sequences that happen to contain the motif
        while MOTIF in sequence:
            sequence = generate_random_sequence(SEQUENCE_LENGTH)
        label = 0

    sequences.append(sequence)
    labels.append(label)

print("Example positive sequence:", sequences[0])
print("Corresponding label:      ", labels[0])
print("\nExample negative sequence:", sequences[1])
print("Corresponding label:      ", labels[1])

### Cell 3: Convert DNA Sequences into a 3D Numerical Tensor

Neural networks cannot directly process text characters (`A`, `T`, `G`, `C`). We convert each sequence into a `(30, 4)` matrix using one-hot encoding, producing an overall dataset tensor `X` of shape `(2000, 30, 4)`.

In [ ]:
base_to_index = {
    "A": 0,
    "T": 1,
    "G": 2,
    "C": 3
}


def encode_sequence(sequence):
    encoded = np.zeros((SEQUENCE_LENGTH, 4), dtype=np.float32)
    for position, base in enumerate(sequence):
        encoded[position, base_to_index[base]] = 1.0
    return encoded


X = np.array([encode_sequence(seq) for seq in sequences])
y = np.array(labels)

print("Input tensor shape:", X.shape)
print("Target shape:      ", y.shape)

#### Classroom Checkpoint: Inspecting One Sample
Pause here and ask the class:
- **What does `2000` represent?** *(Number of sequences / samples)*
- **What does `30` represent?** *(Number of nucleotide positions / timesteps per sequence)*
- **What does `4` represent?** *(Four possible nucleotide features at each position)*

Let's inspect the first sequence and its `(30, 4)` one-hot matrix:

In [ ]:
print("Original sequence:")
print(sequences[0])

print("\nEncoded representation (first 10 positions):")
print(X[0][:10])

print("\nShape of one sample:", X[0].shape)

### Cell 4: Split Training and Testing Data

We split our dataset into **80% training (`1,600` sequences)** and **20% testing (`400` sequences)** using stratified sampling so both sets keep a 50/50 class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training input shape:", X_train.shape)
print("Testing input shape: ", X_test.shape)

### Cell 5: Build the LSTM Architecture

#### Complete Visual Architecture Flow
```text
             BIOLOGICAL DATA
                    │
              DNA Sequences
                    │
             One-hot Encoding
                    │
             3D Input Tensor
              (2000, 30, 4)
                    │
               LSTM Layer
               (32 units)
                    │
             Dropout (0.2)
                    │
              Dense Layer
               (16 units)
                    │
             Sigmoid Output
                    │
            Probability (0–1)
                    │
             Classification
          Motif Present / Absent
```

#### Layer Breakdown
| Layer | Purpose |
| :--- | :--- |
| `Input(shape=(30, 4))` | Receives a single DNA sequence tensor (`30` timesteps, `4` features) |
| `LSTM(units=32)` | Learns sequential dependencies across the 30 positions |
| `Dropout(0.2)` | Randomly drops 20% of connections during training to reduce overfitting |
| `Dense(16, activation="relu")` | Learns non-linear combinations of the LSTM's learned features |
| `Dense(1, activation="sigmoid")` | Outputs a probability between `0` and `1` for motif presence |

> **Important Teaching Point:** `units=32` means the LSTM has **32 hidden memory units**—it does *not* mean the sequence has 32 nucleotides. Also, `return_sequences=False` tells the LSTM to return only its final summary vector `(batch_size, 32)` rather than an output at every single nucleotide position.

In [ ]:
model = Sequential([
    Input(shape=(30, 4)),
    LSTM(units=32, return_sequences=False),
    Dropout(0.2),
    Dense(units=16, activation="relu"),
    Dense(units=1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

#### Optional Demo: `return_sequences=True` vs. `return_sequences=False`
Run the cell below to show students how `return_sequences=True` outputs a 3D tensor `(1, 30, 32)` with 32 features at *every* timestep, whereas `return_sequences=False` outputs a 2D tensor `(1, 32)` representing the final sequence summary.

In [ ]:
sample_input = tf.random.normal((1, 30, 4))

demo_lstm_seq = LSTM(units=32, return_sequences=True)
demo_lstm_last = LSTM(units=32, return_sequences=False)

print("With return_sequences=True: ", demo_lstm_seq(sample_input).shape)
print("With return_sequences=False:", demo_lstm_last(sample_input).shape)

### Cell 6: Train the LSTM Model

| Parameter | Meaning |
| :--- | :--- |
| `epochs=15` | Number of complete passes through the training dataset |
| `batch_size=32` | Number of sequences processed before updating model weights |
| `validation_split=0.2` | 20% of the training set (`320` sequences) is held out to monitor validation performance |

> **Important Distinction:** **LSTM** is the neural network architecture; **Adam** is the optimization algorithm used to update its weights via backpropagation through time.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

### Cell 7: Visualize Training & Validation Curves

- **Accuracy** should increase as the model learns to detect `ATGCGT`.
- **Loss** should decrease over epochs.
- A large gap between training and validation curves indicates **overfitting**.

In [ ]:
plt.figure(figsize=(12, 4))

# Accuracy subplot
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy", marker="o")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", marker="s")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

# Loss subplot
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss", marker="o")
plt.plot(history.history["val_loss"], label="Validation Loss", marker="s")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Cell 8: Evaluate on Unseen Test Data

Now we evaluate our trained model on the `400` unseen sequences in `X_test` and inspect the **classification report** and **confusion matrix**.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}\n")

# Generate class predictions
predictions = model.predict(X_test, verbose=0)
predicted_classes = (predictions >= 0.5).astype(int).flatten()

print("Classification Report:")
print(classification_report(
    y_test,
    predicted_classes,
    target_names=["Motif Absent (0)", "Motif Present (1)"]
))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, predicted_classes)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Absent", "Present"],
    yticklabels=["Absent", "Present"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("LSTM Confusion Matrix")
plt.show()

### Cell 9: Test Your Own Custom DNA Sequences!

Try passing your own 30-nucleotide sequences into `predict_motif()`—place `ATGCGT` at the beginning, middle, or end to verify that the LSTM detects it regardless of position!

In [ ]:
def predict_motif(sequence):
    sequence = sequence.upper().strip()
    if len(sequence) != SEQUENCE_LENGTH:
        raise ValueError(f"Sequence must be exactly {SEQUENCE_LENGTH} bases long (got {len(sequence)}).")

    encoded = encode_sequence(sequence)
    # Add batch dimension: (30, 4) -> (1, 30, 4)
    encoded = np.expand_dims(encoded, axis=0)

    probability = model.predict(encoded, verbose=0)[0][0]
    prediction = "Motif Present" if probability >= 0.5 else "Motif Absent"

    print(f"Sequence:    {sequence}")
    print(f"Probability: {probability:.4f}")
    print(f"Prediction:  {prediction}\n")


# 1. Motif at the start
predict_motif("ATGCGTACGTACGTACGTACGTACGTACGT")

# 2. Motif near the end
predict_motif("AAAAAAAAAATTTTTTTTTTATGCGTCCCC")

# 3. Motif absent
predict_motif("AAAAAAAAAATTTTTTTTTTCCCCCCCCCC")

---
## 4. Classroom Discussion & Genomics Wrap-Up

### Five Quick Review Questions for Students
1. **Why does LSTM need sequential data?**
   *Because it processes ordered steps sequentially to learn patterns and dependencies across positions.*
2. **Why do we use one-hot encoding?**
   *To convert categorical nucleotides (`A`, `T`, `G`, `C`) into numerical vectors without imposing an artificial numeric order.*
3. **What does `(2000, 30, 4)` represent?**
   *`2000` samples (sequences), `30` timesteps (positions), and `4` features per timestep.*
4. **What is the role of the forget gate?**
   *To decide which information from earlier timesteps should be discarded from internal memory.*
5. **What is the difference between LSTM and Adam?**
   *LSTM is the neural network layer/architecture; Adam is the optimization algorithm that updates the model's weights.*

### Connecting to Real Genomics Applications
| Application | What LSTM Can Learn |
| :--- | :--- |
| **DNA Sequence Classification** | Regulatory motifs and positional sequence patterns |
| **RNA Sequence Analysis** | Splice-site signals and nucleotide dependencies |
| **Gene Expression Time Series** | Dynamic changes in expression levels across timepoints |
| **Protein Sequence Analysis** | Secondary structure motifs across amino acid chains |
| **Longitudinal Clinical Data** | Patient biomarker trajectories over hospital visits |

> **Real-World Caveat:** This demonstration uses short, fixed-length synthetic sequences. Real genomic workflows require handling variable sequence lengths, reference genome splits (to avoid data leakage), class imbalance, and benchmarking against **1D CNNs** and **Transformer-based** architectures.